
<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 15px; height: 80px">

# Random Forests and ExtraTrees


_Authors: Matt Brems (DC), Riley Dallas (AUS)_

---

## Random Forests

The Random Forest is one of the most powerful machine learning algorithms available today. It is a supervised machine learning algorithm that can be used for both classification and regression.

With bagged decision trees, we generate many different trees on pretty similar data. These trees are **highly correlated** with one another. 

If two trees are highly correlated, then the variance will be about as high as we'd see with individual decision trees. 

By de-correlating our trees from one another, we can drastically reduce the variance of our model.

This change will reduce our variance (at the expense of a small increase in bias). This should often improve our model's performance.

So how do we de-correlate our trees?

![](assets/images/forest.jpeg)

Random forests differ from bagging decision trees in only one way: they use a modified tree learning algorithm that selects, at each split in the learning process, a **random subset of the features**. This process is sometimes called the *random subspace method*. This way the trees built with the training data can be quite different from each other.

The reason for doing this is the correlation of the trees in an ordinary bootstrap sample: if one or a few features are very strong predictors for the response variable (target output), these features will be used in many/all of the bagged decision trees, causing them to become correlated. By selecting a random subset of features at each split, we counter this correlation between base trees, strengthening the overall model.


For a problem with $p$ features, it is typical to use:

- $\sqrt{p}$ (rounded down) features in each split for a classification problem.
- $p/3$ (rounded down) with a minimum node size of 5 as the default for a regression problem.

While this is a guideline, Hastie and Tibshirani (authors of Introduction to Statistical Learning and Elements of Statistical Learning) have suggested this as a good rule in the absence of some rationale to do something different.

Random forests, a step beyond bagged decision trees, are **very widely used** classifiers and regressors. They are relatively simple to use because they require very few parameters to set and they perform pretty well.
- It is quite common for interviewers to ask how a random forest is constructed or how it is superior to a single decision tree.

--- 

## Picture Worth A Thousand Words

![image](https://miro.medium.com/max/720/1*5vlUF8FRR6flPPWK4wt-Kw.png)

[Source](https://medium.com/@harshdeepsingh_35448/understanding-random-forests-aa0ccecdbbbb)

## Extremely Randomized Trees (ExtraTrees) 🌲🌴🌳
Adding another step of randomization (and thus de-correlation) yields extremely randomized trees, or _ExtraTrees_. Like Random Forests, these are trained using the random subspace method (sampling of features). However, they are trained on the entire dataset instead of bootstrapped samples. A layer of randomness is introduced in the way the nodes are split. Instead of computing the locally optimal feature/split combination (based on, e.g., information gain or the Gini impurity) for each feature under consideration, a random value is selected for the split. This value is selected from the feature's empirical range.

This further reduces the variance, but causes an increase in bias. If you're considering using ExtraTrees, you might consider this to be a hyperparameter you can tune. Build an ExtraTrees model and a Random Forest model, then compare their performance!

That's exactly what we'll do below.

## Import libraries
---

We'll need the following libraries for today's lecture:
- `pandas`
- `numpy`
- `GridSearchCV`, `train_test_split` and `cross_val_score` from `sklearn`'s `model_selection` module 
- `RandomForestClassifier` and `ExtraTreesClassifier` from `sklearn`'s `ensemble` module 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer

## Load Data
---

Load `train.csv` and `test.csv` from Kaggle into `DataFrames`.

## Data Cleaning: Drop the two rows with missing `Embarked` values from train
---

## Data Cleaning: `Fare`
---

The test set has one row with a missing value for `Fare`. Fill it with the average `Fare` with everyone from the same `Pclass`. **Use the training set to calculate the average!**

## Data Cleaning: `Age`
---

Let's simply impute all missing ages to the mean of the training age using SimpleImputer.

**NOTE**: 
1. We could use `.fillna()`, though I'd like to show an example of SimpleImputer (and how I'm sure many of you wished you knew about this for P2).
2. There can be other methods to consider (such as IterativeImputer), but we'll do it this way as a matter of convenience.

In [ ]:
# What if there were multiple numerical columns you wanted to do this to? (Like in P2!)

# Get all numerical columns

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

In [ ]:

# Fill missing values in categorical columns using most frequent value
cat_columns = ['Embarked']
cat_imputer = SimpleImputer(strategy='most_frequent')
train_df[cat_columns] = cat_imputer.fit_transform(train_df[cat_columns])
test_df[cat_columns] = cat_imputer.transform(test_df[cat_columns])

# Create a column for each categorical feature using one-hot encoding
categorical_features = ['Pclass', 'Sex', 'Embarked']
encoder = OneHotEncoder(drop='first', sparse=False)
train_encoded_features = pd.DataFrame(encoder.fit_transform(train_df[categorical_features]))
test_encoded_features = pd.DataFrame(encoder.transform(test_df[categorical_features]))
train_encoded_features.columns = encoder.get_feature_names(categorical_features)
test_encoded_features.columns = encoder.get_feature_names(categorical_features)
train_df = pd.concat([train_df.drop(categorical_features, axis=1), train_encoded_features], axis=1)
test_df = pd.concat([test_df.drop(categorical_features, axis=1), test_encoded_features], axis=1)

# Check the cleaned datasets
print("Cleaned Training Set:")
print(train_df.head())
print("\nCleaned Test Set:")
print(test_df.head())

In [ ]:
# Fill missing values in numerical columns using mean
num_columns = ['Age', 'Fare']
num_imputer = SimpleImputer(strategy='mean')
df[num_columns] = num_imputer.fit_transform(df[num_columns])

# Fill missing values in categorical columns using most frequent value
cat_columns = ['Embarked']
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_columns] = cat_imputer.fit_transform(df[cat_columns])

# Create a column for each categorical feature using one-hot encoding
categorical_features = ['Pclass', 'Sex', 'Embarked']
encoder = OneHotEncoder(drop='first', sparse=False)
encoded_features = pd.DataFrame(encoder.fit_transform(df[categorical_features]))
encoded_features.columns = encoder.get_feature_names(categorical_features)
df = pd.concat([df.drop(categorical_features, axis=1), encoded_features], axis=1)

## Feature Engineering: `Cabin`
---

Since there are so many missing values for `Cabin`, let's binarize that column as follows:
- 1 if there originally was a value for `Cabin`
- 0 if it was null

**Do this for both `train` and `test`**

## Feature Engineering: Dummies
---

Dummy the `Sex` and `Embarked` columns. Be sure to set `drop_first=True`.

## Model Prep: Create `X` and `y` variables
---

Our features will be:
- `Pclass`
- `Age`
- `SibSp`
- `Parch`
- `Fare`
- `Cabin`
- `Sex_male`
- `Embarked_Q`
- `Embarked_S`

And our target will be `Survived`

## Challenge: What is our baseline accuracy?
---

The baseline accuracy is the percentage of the majority class, regardless of whether it is 1 or 0. It serves as the benchmark for our model to beat.

## Train/Test Split
---

I know it can be confusing having an `X_test` from our training data vs a test set from Kaggle. If you want, you can use `X_val`/`y_val` for what we normally call `X_test`/`y_test`.

## Model instantiation
---

Create an instance of `RandomForestClassifier` and `ExtraTreesClassifier`.

## Model Evaluation
---

Which one has a higher `cross_val_score`?

## Grid Search
---

They're both pretty close performance-wise. We could Grid Search over both, but for the sake of time we'll go with `RandomForestClassifier`.

## Feature Importance
---

In [ ]:
#???


In [ ]:
#Ahhh, I see.


In [ ]:
#Let's make it look nice!


### Wrap it up in a function

## Can we visualize one of these trees?
---

## Kaggle Submission
---

Now that we've evaluated our model, let's submit our predictions to [Kaggle](https://www.kaggle.com/competitions/titanic/overview).